In [ ]:
import geopandas as gpd
import numpy as np
from shapely.geometry import box

In [ ]:
base = gpd.read_file("/g/data/jk72/ae8456/epsg3031_regions.shp")

# Ensure CRS is EPSG:3031
base = base.to_crs("EPSG:3031")

minx, miny, maxx, maxy = base.total_bounds

In [ ]:
grid_size = 250  # 10 km

xmin, xmax = 0, 540300
ymin, ymax = -2058481, -1234867

xs = np.arange(xmin, xmax, grid_size)
ys = np.arange(ymin, ymax, grid_size)

from shapely.geometry import box

polygons = [
    box(x, y, x + grid_size, y + grid_size)
    for x in xs
    for y in ys
]

grid = gpd.GeoDataFrame(geometry=polygons, crs="EPSG:3031")

print(f"Grid cells: {len(grid)}")

In [ ]:
import geopandas as gpd

# Load regions (14 polygons)
regions = gpd.read_file("/g/data/jk72/ae8456/epsg3031_regions.shp")

# Load swaths (GeoJSON)
swaths = gpd.read_file(
    "/g/data/jk72/ae8456/SWOT/KaRIn_2kms_science_geometries.geojson"
)

# Reproject swaths to match regions
swaths = swaths.to_crs(regions.crs)

orbit_col = "pass_number"  # adjust if needed

In [ ]:
swaths = swaths.dissolve(by=orbit_col, as_index=False)
swaths = swaths.sort_values(orbit_col).reset_index(drop=True)
relevant_passes = [4, 6, 8, 9, 11, 13,  32,  34,  36,  37,  39,  41,  60,
        62,  64,  65,  67,  69,  88,  90,  92,  93,  95,  97, 116, 118,
       120, 121, 123, 125, 144, 146, 149, 151, 172, 174, 177, 179, 200,
       202, 205, 207, 228, 230, 233, 235, 254, 256, 258, 259, 261, 263,
       282, 284, 286, 287, 289, 291, 310, 312, 314, 315, 317, 319, 338,
       340, 342, 343, 345, 347, 366, 368, 370, 371, 373, 375, 394, 396,
       398, 399, 401, 403, 422, 424, 427, 429, 431, 450, 452, 455, 457,
       478, 480, 483, 485, 506, 508, 511, 513, 532, 534, 536, 537, 539,
       541, 560, 562, 564, 565, 567, 569]
swaths = swaths[swaths[orbit_col].isin(relevant_passes)]

In [ ]:
swaths

In [ ]:
n = len(swaths)

subsets = {
    "first_third":  swaths.iloc[: n//3],
    "second_third": swaths.iloc[n//3 : 2*n//3],
    "third_third":  swaths.iloc[2*n//3 :],
    #"first_half":   swaths.iloc[: n//2],
    #"second_half":  swaths.iloc[n//2 :],
    #"all":          swaths
}

In [ ]:
swaths

In [ ]:
import os
import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin

In [ ]:
outdir = "/g/data/jk72/ae8456/SWOT/grids/Victoria_Land"
os.makedirs(outdir, exist_ok=True)

# --- Get grid resolution and bounds ---
xmin, ymin, xmax, ymax = grid.total_bounds
res_x = grid.geometry.iloc[0].bounds[2] - grid.geometry.iloc[0].bounds[0]
res_y = grid.geometry.iloc[0].bounds[3] - grid.geometry.iloc[0].bounds[1]

width  = int((xmax - xmin) / res_x)
height = int((ymax - ymin) / res_y)

transform = from_origin(xmin, ymax, res_x, res_y)

subsets = {
    "first_third":  swaths.iloc[: n//3],
    "second_third": swaths.iloc[n//3 : 2*n//3],
    "third_third":  swaths.iloc[2*n//3 :],
    "first_half":   swaths.iloc[: n//2],
    "second_half":  swaths.iloc[n//2 :],
    "all":          swaths
}

for name, subset in subsets.items():
    print(f"Processing {name}...")

    colname = f"{name}_coverage"

    grid[colname] = grid.geometry.apply(
        lambda cell: subset[subset.geometry.intersects(cell)][orbit_col].nunique()
    )

    shapes = ((geom, value) for geom, value in zip(grid.geometry, grid[colname]))

    raster = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype="float32"
    )

    outpath = os.path.join(outdir, f"{name}_coverage_250m.tif")

    with rasterio.open(
        outpath,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="float32",
        crs=grid.crs,
        transform=transform,
    ) as dst:
        dst.write(raster, 1)

    print(f"Saved {outpath}")

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import matplotlib.colors as colors
import matplotlib.colors as mcolors
import numpy as np

overlay = gpd.read_file(
    "/g/data/jk72/ae8456/SWOT/Coastline/add_coastline_high_res_polygon_v7_6.shp"
).to_crs(grid.crs)


cmap = plt.get_cmap("plasma")  # continuous version
norm = mcolors.Normalize(
    vmin=grid["all_coverage"].min(),
    vmax=grid["all_coverage"].max()
)

fig, ax = plt.subplots(figsize=(14, 14))

grid.plot(
    ax=ax,
    column="all_coverage",
    cmap=cmap,
    norm=norm,
    legend=True,

)

overlay.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=1
)
xmin, xmax = 0, 540300     
ymin, ymax = -2058481, -1234867   

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_title("Full-Cycle coverage")
ax.set_axis_off()

plt.show()
